# RT-FSAS — Improved GNN training (Colab)

**Prerequisites on Google Drive**

- Clone or copy the repo so you have a folder that contains `src/` and `setup/`.
- Default path used below: `/content/drive/MyDrive/ML_project/project`
- Pre-built graph files: `graphs/la_liga_2015_16_train.pt` and `graphs/la_liga_2015_16_full_shards/`

**Run cells in order.** If `train_gnn.py` in your Drive copy does not support `--use_class_weights`, pull the latest code from the repo or sync `src/training/train_gnn.py` from your machine.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys

# Repo root that contains src/ and setup/ (edit if your Drive layout differs)
PROJECT_ROOT = "/content/drive/MyDrive/ML_project/project"

assert os.path.isdir(PROJECT_ROOT), f"Missing: {PROJECT_ROOT} — fix PROJECT_ROOT or clone the repo into Drive."
assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), "src/ not found: PROJECT_ROOT is wrong."

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("CWD:", os.getcwd())
print("src:", os.path.isdir("src"))

## Install dependencies
Run after the cell above so `setup/requirements.txt` resolves. Re-run after **Runtime → Restart session** if needed.

In [ ]:
!python -m pip install --upgrade pip
!pip install -r setup/requirements.txt
!pip install ijson torch-geometric

In [ ]:
# Optional: if torch-geometric import fails, install PyG wheels for your torch version (uncomment and adjust URL from https://data.pyg.org/whl/)
# !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.2.0+cpu.html
# !pip install torch-geometric

In [ ]:
!pwd
!ls -la
!ls graphs
!ls checkpoints

## 1) Baseline training (reference run)

In [ ]:
!python -m src.training.train_gnn \
  --dataset_path graphs/la_liga_2015_16_train.pt \
  --checkpoint_path checkpoints/gnn_baseline.pt \
  --history_path checkpoints/gnn_history_baseline.pt \
  --epochs 60 \
  --patience 12 \
  --batch_size 64 \
  --lr 1e-3 \
  --hidden_dim 64 \
  --embed_dim 128 \
  --seed 42

## 2) Improved training (class weights + LR scheduler + label smoothing)

In [ ]:
!python -m src.training.train_gnn \
  --dataset_path graphs/la_liga_2015_16_train.pt \
  --checkpoint_path checkpoints/gnn_weighted.pt \
  --history_path checkpoints/gnn_history_weighted.pt \
  --epochs 60 \
  --patience 12 \
  --batch_size 64 \
  --lr 5e-4 \
  --hidden_dim 96 \
  --embed_dim 128 \
  --seed 42 \
  --use_class_weights \
  --scheduler plateau \
  --scheduler_factor 0.5 \
  --scheduler_patience 3 \
  --label_smoothing 0.05

## 3) Same improved recipe, different seed (pick best val metrics)

In [ ]:
!python -m src.training.train_gnn \
  --dataset_path graphs/la_liga_2015_16_train.pt \
  --checkpoint_path checkpoints/gnn_weighted_seed7.pt \
  --history_path checkpoints/gnn_history_weighted_seed7.pt \
  --epochs 60 \
  --patience 12 \
  --batch_size 64 \
  --lr 5e-4 \
  --hidden_dim 96 \
  --embed_dim 128 \
  --seed 7 \
  --use_class_weights \
  --scheduler plateau \
  --scheduler_factor 0.5 \
  --scheduler_patience 3 \
  --label_smoothing 0.05

## 4) Build FAISS index
Point `--checkpoint_path` to whichever run had the best validation loss. Default below uses the improved weighted run.

In [ ]:
!python -m src.retrieval.build_index \
  --graphs_path graphs/la_liga_2015_16_full_shards \
  --checkpoint_path checkpoints/gnn_weighted.pt \
  --index_dir index

## 5) Optional: retrieval smoke test

In [ ]:
import torch
from src.retrieval.retriever import TacticalRetriever

r = TacticalRetriever(index_dir="index")
q = torch.randn(128)
print(r.retrieve(q, k=3))